# 02 - Treinamento e avaliação dos modelos

> **Escreva aqui, com suas palavras, o objetivo deste notebook.**
> As células de markdown marcadas com `>` são espaços para você preencher.
> Elas fazem parte da defesa de autoria do trabalho e não devem ficar em branco.

## Ordem de execução

As hipóteses são avaliadas nesta ordem, e a ordem não é negociável:

1. **H1** - desempenho preditivo, no conjunto de **validação**
2. **H2** - custo computacional e complexidade, no conjunto de **validação**
3. **Congelamento** dos resultados
4. **H3** - generalização, no conjunto **independente (Hannousse)**, uma única vez

O conjunto independente não é consultado antes do passo 4. Observar seu
resultado antes de fechar H1 e H2 transformaria qualquer ajuste posterior em
sintonia sobre o conjunto de teste, e as hipóteses deixariam de ser
pré-registradas.

In [44]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

import sys
sys.path.insert(0, str(Path.cwd().parent / "src"))
from avaliacao import avaliar_completo, avaliar_predicao, ordens_de_magnitude

PROC = Path("../dados/processados")
RESULTADOS = Path("../resultados")
TABELAS = RESULTADOS / "tabelas"
TABELAS.mkdir(parents=True, exist_ok=True)

SEMENTE = 42
np.random.seed(SEMENTE)

## 1. Carregamento dos dados

> Descreva de onde vêm estes arquivos e o que cada partição representa.

In [45]:
def carregar(nome):
    return pd.read_csv(PROC / f"{nome}.csv", index_col=0)

X_tr    = carregar("X_tr")
X_val   = carregar("X_val")
X_teste = carregar("X_teste")

y_tr    = carregar("y_tr")["y"]
y_val   = carregar("y_val")["y"]
y_teste = carregar("y_teste")["y"]

print(f"treino     {X_tr.shape}   phishing: {y_tr.mean():.3f}")
print(f"validação  {X_val.shape}   phishing: {y_val.mean():.3f}")
print(f"teste ind. {X_teste.shape}   phishing: {y_teste.mean():.3f}")

treino     (188296, 37)   phishing: 0.427
validação  (47074, 37)   phishing: 0.427
teste ind. (11385, 37)   phishing: 0.498


In [46]:
# Verificações que devem passar antes de qualquer treino.
assert list(X_tr.columns) == list(X_val.columns) == list(X_teste.columns), \
    "As três partições precisam ter os mesmos atributos na mesma ordem."
assert not X_tr.isna().any().any(), "Há valores ausentes no treino."
assert not X_val.isna().any().any(), "Há valores ausentes na validação."
assert not X_teste.isna().any().any(), "Há valores ausentes no teste independente."
assert set(y_tr.unique()) <= {0, 1}, "Rótulos fora da convenção binária."

print(f"{X_tr.shape[1]} atributos, verificações aprovadas.")

37 atributos, verificações aprovadas.


## 2. Definição dos modelos

> Justifique aqui a escolha dos quatro algoritmos e a exclusão do SVM com
> kernel RBF.

O desbalanceamento é tratado exclusivamente por ponderação de classes durante
o treino. Nenhuma reamostragem é aplicada, e nenhuma das partições de
avaliação é rebalanceada.

In [47]:
# Proporção usada na ponderação do XGBoost (negativos / positivos).
peso_positivo = float((y_tr == 0).sum() / (y_tr == 1).sum())
print(f"scale_pos_weight = {peso_positivo:.4f}")

modelos = {
    "Regressão Logística": Pipeline([
        ("escala", StandardScaler()),
        ("clf", LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=SEMENTE,
        )),
    ]),
    "Árvore de Decisão": DecisionTreeClassifier(
        class_weight="balanced",
        random_state=SEMENTE,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=SEMENTE,
        n_jobs=1,          # fixado para que a medição de tempo seja comparável
    ),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        scale_pos_weight=peso_positivo,
        random_state=SEMENTE,
        n_jobs=1,
        tree_method="hist",
        eval_metric="logloss",
    ),
}

for nome in modelos:
    print(nome)

scale_pos_weight = 1.3415
Regressão Logística
Árvore de Decisão
Random Forest
XGBoost


### Nota sobre a padronização

A Regressão Logística é o único modelo em Pipeline com `StandardScaler`.
Modelos baseados em árvore particionam por limiar em cada atributo
isoladamente e são invariantes a transformações monotônicas de escala;
padronizá-los não mudaria as partições e apenas acrescentaria custo.

O `StandardScaler` é ajustado dentro do Pipeline, portanto suas estatísticas
vêm apenas do conjunto de treino. Ajustá-lo sobre os dados completos
vazaria informação das partições de avaliação.

## 3. Treinamento

> Registre aqui o tempo de treino observado e qualquer observação sobre a
> execução.

In [48]:
tempos_treino = {}

for nome, modelo in modelos.items():
    inicio = time.perf_counter()
    modelo.fit(X_tr, y_tr)
    tempos_treino[nome] = time.perf_counter() - inicio
    print(f"{nome:22s} {tempos_treino[nome]:7.2f}s")

Regressão Logística       2.31s
Árvore de Decisão         1.23s
Random Forest            17.89s
XGBoost                   2.35s


## 4. H1 - desempenho preditivo

**Hipótese:** os modelos de ensemble superam os modelos de base em F1.

A avaliação é feita no conjunto de validação, extraído do PhiUSIIL. A classe
positiva é phishing.

> Escreva aqui sua leitura dos resultados depois de executar a célula.

In [49]:
registros = [
    avaliar_completo(nome, modelo, X_val, y_val)
    for nome, modelo in modelos.items()
]

res_val = pd.DataFrame(registros).set_index("modelo")
res_val["tempo_treino_s"] = pd.Series(tempos_treino)

colunas_h1 = ["f1", "precisao", "recall", "roc_auc", "pr_auc", "vp", "fp", "fn"]
res_val[colunas_h1].round(4)

,f1,precisao,recall,roc_auc,pr_auc,vp,fp,fn
modelo,,,,,,,,
Regressão Logística,0.9951,0.9989,0.9914,0.9980,0.9986,19931,22,173
Árvore de Decisão,0.9957,0.9964,0.9950,0.9963,0.9943,20004,72,100
Random Forest,0.9969,0.9989,0.9949,0.9984,0.9984,20002,22,102
XGBoost,0.9970,0.9994,0.9945,0.9983,0.9988,19994,12,110


In [50]:
# Verificação formal da H1.
BASE = ["Regressão Logística", "Árvore de Decisão"]
ENSEMBLE = ["Random Forest", "XGBoost"]

f1_base_max = res_val.loc[BASE, "f1"].max()
f1_ens_max = res_val.loc[ENSEMBLE, "f1"].max()

print(f"melhor F1 entre os modelos de base:      {f1_base_max:.4f}")
print(f"melhor F1 entre os modelos de ensemble:  {f1_ens_max:.4f}")
print(f"ganho absoluto:                          {f1_ens_max - f1_base_max:+.4f}")
print()
print("H1 corroborada." if f1_ens_max > f1_base_max else "H1 NÃO corroborada.")

melhor F1 entre os modelos de base:      0.9957
melhor F1 entre os modelos de ensemble:  0.9970
ganho absoluto:                          +0.0012

H1 corroborada.


## 5. H2 - custo computacional e complexidade

**Hipótese:** o ganho de F1 dos ensembles é inferior a 0,05 em valor absoluto,
enquanto o tempo de inferência e o número de nós aumentam em pelo menos uma
ordem de magnitude.

O limiar de 0,05 e o critério de uma ordem de magnitude foram fixados antes da
observação dos resultados e não são revisáveis.

> Comente aqui o que a comparação revela sobre a relação entre custo e ganho.

In [51]:
colunas_h2 = [
    "f1", "tempo_mediana_us", "tempo_desvio_us",
    "n_arvores", "n_nos_total", "profundidade_max", "tipo_complexidade",
]
res_val[colunas_h2].round(4)

,f1,tempo_mediana_us,tempo_desvio_us,n_arvores,n_nos_total,profundidade_max,tipo_complexidade
modelo,,,,,,,
Regressão Logística,0.9951,0.3329,0.0587,0,37,0,coeficientes
Árvore de Decisão,0.9957,0.2308,0.0079,1,2649,51,nos_arvore
Random Forest,0.9969,11.1707,0.1584,100,277014,49,nos_arvore
XGBoost,0.9970,1.7712,0.1321,100,4156,6,nos_arvore


In [52]:
# Verificação formal da H2, usando a Árvore de Decisão como referência
# de complexidade: é o modelo de base que possui nós, o que torna a
# comparação de número de nós uma comparação entre grandezas equivalentes.
referencia = "Árvore de Decisão"
melhor_ensemble = res_val.loc[ENSEMBLE, "f1"].idxmax()

ganho_f1 = res_val.loc[melhor_ensemble, "f1"] - f1_base_max
om_tempo = ordens_de_magnitude(
    res_val.loc[melhor_ensemble, "tempo_mediana_us"],
    res_val.loc[referencia, "tempo_mediana_us"],
)
om_nos = ordens_de_magnitude(
    res_val.loc[melhor_ensemble, "n_nos_total"],
    res_val.loc[referencia, "n_nos_total"],
)

cond_ganho = abs(ganho_f1) < 0.05
cond_tempo = om_tempo >= 1.0
cond_nos = om_nos >= 1.0

print(f"modelo avaliado: {melhor_ensemble}  (referência: {referencia})")
print(f"  ganho de F1              {ganho_f1:+.4f}   < 0,05 ? {cond_ganho}")
print(f"  tempo, ordens de magn.   {om_tempo:+.2f}     >= 1  ? {cond_tempo}")
print(f"  nós, ordens de magnitude {om_nos:+.2f}     >= 1  ? {cond_nos}")
print()
print("H2 corroborada." if (cond_ganho and cond_tempo and cond_nos)
      else "H2 NÃO corroborada em todos os critérios.")

modelo avaliado: XGBoost  (referência: Árvore de Decisão)
  ganho de F1              +0.0012   < 0,05 ? True
  tempo, ordens de magn.   +0.88     >= 1  ? False
  nós, ordens de magnitude +0.20     >= 1  ? False

H2 NÃO corroborada em todos os critérios.


## 6. Congelamento dos resultados internos

A partir deste ponto, nenhum hiperparâmetro é alterado e nenhum modelo é
retreinado. O conjunto independente é consultado a seguir, uma única vez.

In [53]:
res_val.to_csv(TABELAS / "resultados_validacao.csv")

registro_h1_h2 = {
    "semente": SEMENTE,
    "n_treino": int(len(X_tr)),
    "n_validacao": int(len(X_val)),
    "n_atributos": int(X_tr.shape[1]),
    "f1_base_max": float(f1_base_max),
    "f1_ensemble_max": float(f1_ens_max),
    "melhor_ensemble": melhor_ensemble,
    "h1_corroborada": bool(f1_ens_max > f1_base_max),
    "h2_ganho_f1": float(ganho_f1),
    "h2_om_tempo": float(om_tempo),
    "h2_om_nos": float(om_nos),
    "h2_corroborada": bool(cond_ganho and cond_tempo and cond_nos),
}

with open(TABELAS / "hipoteses_h1_h2.json", "w", encoding="utf-8") as f:
    json.dump(registro_h1_h2, f, indent=2, ensure_ascii=False)

print(json.dumps(registro_h1_h2, indent=2, ensure_ascii=False))

{
  "semente": 42,
  "n_treino": 188296,
  "n_validacao": 47074,
  "n_atributos": 37,
  "f1_base_max": 0.9957192633150821,
  "f1_ensemble_max": 0.9969583644976315,
  "melhor_ensemble": "XGBoost",
  "h1_corroborada": true,
  "h2_ganho_f1": 0.0012391011825494402,
  "h2_om_tempo": 0.8849753118840838,
  "h2_om_nos": 0.19559358058790877,
  "h2_corroborada": false
}


## 7. H3 - generalização para conjunto independente

**Hipótese:** o F1 cai 0,10 ou mais em valor absoluto quando os modelos são
aplicados ao conjunto de Hannousse e Yahiouche.

Esta é a primeira e única consulta ao conjunto independente. Os modelos foram
treinados exclusivamente no PhiUSIIL e não recebem qualquer ajuste aqui.

> Escreva aqui, antes de executar, o que você espera encontrar. Registrar a
> expectativa prévia fortalece a interpretação do resultado, qualquer que ele
> seja.

In [54]:
registros_teste = [
    {"modelo": nome, **avaliar_predicao(modelo, X_teste, y_teste)}
    for nome, modelo in modelos.items()
]

res_teste = pd.DataFrame(registros_teste).set_index("modelo")
res_teste[["f1", "precisao", "recall", "roc_auc", "pr_auc"]].round(4)

,f1,precisao,recall,roc_auc,pr_auc
modelo,,,,,
Regressão Logística,0.6650,0.4989,0.9972,0.6547,0.6007
Árvore de Decisão,0.6651,0.4989,0.9977,0.5013,0.4989
Random Forest,0.6650,0.4988,0.9974,0.5010,0.4987
XGBoost,0.6655,0.4991,0.9984,0.6339,0.5808


In [55]:
comparacao = pd.DataFrame({
    "f1_validacao": res_val["f1"],
    "f1_teste_independente": res_teste["f1"],
})
comparacao["queda_absoluta"] = (
    comparacao["f1_validacao"] - comparacao["f1_teste_independente"]
)
comparacao["h3_por_modelo"] = comparacao["queda_absoluta"] >= 0.10

comparacao.round(4)

,f1_validacao,f1_teste_independente,queda_absoluta,h3_por_modelo
modelo,,,,
Regressão Logística,0.9951,0.6650,0.3301,True
Árvore de Decisão,0.9957,0.6651,0.3306,True
Random Forest,0.9969,0.6650,0.3319,True
XGBoost,0.9970,0.6655,0.3315,True


In [56]:
queda_melhor = float(
    comparacao.loc[melhor_ensemble, "queda_absoluta"]
)
h3_corroborada = queda_melhor >= 0.10

print(f"modelo avaliado: {melhor_ensemble}")
print(f"  F1 na validação            {comparacao.loc[melhor_ensemble, 'f1_validacao']:.4f}")
print(f"  F1 no teste independente   {comparacao.loc[melhor_ensemble, 'f1_teste_independente']:.4f}")
print(f"  queda absoluta             {queda_melhor:.4f}   >= 0,10 ? {h3_corroborada}")
print()
print("H3 corroborada." if h3_corroborada else "H3 NÃO corroborada.")

modelo avaliado: XGBoost
  F1 na validação            0.9970
  F1 no teste independente   0.6655
  queda absoluta             0.3315   >= 0,10 ? True

H3 corroborada.


In [57]:
comparacao.to_csv(TABELAS / "comparacao_validacao_teste.csv")
res_teste.to_csv(TABELAS / "resultados_teste_independente.csv")

registro_h3 = {
    "n_teste_independente": int(len(X_teste)),
    "queda_f1_melhor_ensemble": queda_melhor,
    "h3_corroborada": bool(h3_corroborada),
    "quedas_por_modelo": comparacao["queda_absoluta"].round(4).to_dict(),
}

with open(TABELAS / "hipoteses_h3.json", "w", encoding="utf-8") as f:
    json.dump(registro_h3, f, indent=2, ensure_ascii=False)

print(json.dumps(registro_h3, indent=2, ensure_ascii=False))

{
  "n_teste_independente": 11385,
  "queda_f1_melhor_ensemble": 0.33146685500823736,
  "h3_corroborada": true,
  "quedas_por_modelo": {
    "Regressão Logística": 0.3301,
    "Árvore de Decisão": 0.3306,
    "Random Forest": 0.3319,
    "XGBoost": 0.3315
  }
}


## 8. Síntese

> Escreva aqui a conclusão do experimento: quais hipóteses foram corroboradas,
> quais não foram, e o que isso significa para a questão de pesquisa.
>
> Uma hipótese refutada é um resultado válido e reportável. Não ajuste
> limiares para que os números fechem.

In [58]:
sintese = pd.DataFrame({
    "F1 validação": res_val["f1"],
    "F1 teste ind.": res_teste["f1"],
    "Queda": comparacao["queda_absoluta"],
    "Tempo (µs/URL)": res_val["tempo_mediana_us"],
    "Nós": res_val["n_nos_total"],
}).round(4)

sintese.to_csv(TABELAS / "sintese_final.csv")
sintese

,F1 validação,F1 teste ind.,Queda,Tempo (µs/URL),Nós
modelo,,,,,
Regressão Logística,0.9951,0.6650,0.3301,0.3329,37
Árvore de Decisão,0.9957,0.6651,0.3306,0.2308,2649
Random Forest,0.9969,0.6650,0.3319,11.1707,277014
XGBoost,0.9970,0.6655,0.3315,1.7712,4156


In [59]:
res_teste[["f1", "precisao", "recall", "vp", "fp", "vn", "fn"]].round(4)

,f1,precisao,recall,vp,fp,vn,fn
modelo,,,,,,,
Regressão Logística,0.6650,0.4989,0.9972,5656,5682,31,16
Árvore de Decisão,0.6651,0.4989,0.9977,5659,5685,28,13
Random Forest,0.6650,0.4988,0.9974,5657,5684,29,15
XGBoost,0.6655,0.4991,0.9984,5663,5684,29,9


In [60]:
BRUTOS = Path("../dados/brutos")

phiusiil = pd.read_csv(BRUTOS / "PhiUSIIL_Phishing_URL_Dataset.csv")
hann = pd.read_csv(BRUTOS / "dataset_B_05_2020.csv")

phiusiil_limpo = phiusiil[~phiusiil["URL"].duplicated(keep="first")]
hann_limpo = hann[
    ~hann["url"].duplicated(keep="first")
    & ~hann["url"].isin(set(phiusiil_limpo["URL"]))
]

print(len(phiusiil_limpo), len(hann_limpo))  

235370 11385


In [61]:
print(hann_limpo[["url", "status"]].head(8).to_string())
print()
print(pd.crosstab(hann_limpo["status"].values, y_teste.values))

                                                                                                                              url      status
0                                                                                           http://www.crestonwood.com/router.php  legitimate
1                                                   http://shadetreetechnology.com/V4/validation/a111aedc8ae390eabcfa130e041a10a4    phishing
2  https://support-appleld.com.secureupdate.duilawyeryork.com/ap/89e6a3b4b063b8d/?cmd=_update&dispatch=89e6a3b4b063b8d1b&locale=_    phishing
3                                                                                                              http://rgipt.ac.in  legitimate
4                                                                         http://www.iracing.com/tracks/gateway-motorsports-park/  legitimate
5                                                                                                http://appleid.apple.com-app.es/    phishing
6     

In [62]:
leg_tr = phiusiil_limpo.loc[phiusiil_limpo["label"] == 1, "URL"].head(6).tolist()
for u in leg_tr:
    print("PhiUSIIL leg :", u[:80])

PhiUSIIL leg : https://www.southbankmosaics.com
PhiUSIIL leg : https://www.uni-mainz.de
PhiUSIIL leg : https://www.voicefmradio.co.uk
PhiUSIIL leg : https://www.sfnmjournal.com
PhiUSIIL leg : https://www.rewildingargentina.org
PhiUSIIL leg : https://www.globalreporting.org


In [63]:
comp = pd.DataFrame({
    "phiusiil_leg": X_val[y_val.values == 0].mean(),
    "hannousse_leg": X_teste[y_teste.values == 0].mean(),
})
print(comp.loc[["scheme_is_https", "path_length", "num_tokens_path",
                "url_length", "count_slash", "count_digits"]].round(3))

                 phiusiil_leg  hannousse_leg
scheme_is_https         1.000          0.445
path_length             0.000         20.168
num_tokens_path         0.000          1.368
url_length             27.207         47.391
count_slash             2.000          3.834
count_digits            0.050          1.529


In [64]:
res_val.to_csv(TABELAS / "resultados_validacao.csv")
res_teste.to_csv(TABELAS / "resultados_teste_independente.csv")
comp.to_csv(TABELAS / "comparacao_distribuicao_legitimas.csv")
sintese.to_csv(TABELAS / "sintese_final.csv")

In [65]:
print(res_teste[["f1", "roc_auc", "pr_auc"]].round(4))

                         f1  roc_auc  pr_auc
modelo                                      
Regressão Logística  0.6650   0.6547  0.6007
Árvore de Decisão    0.6651   0.5013  0.4989
Random Forest        0.6650   0.5010  0.4987
XGBoost              0.6655   0.6339  0.5808


In [66]:
espurios = ["path_length", "num_tokens_path", "count_slash",
            "scheme_is_https", "query_length"]
cols = [c for c in X_tr.columns if c not in espurios]

from sklearn.ensemble import RandomForestClassifier
rf2 = RandomForestClassifier(n_estimators=100, class_weight="balanced",
                             random_state=42, n_jobs=1).fit(X_tr[cols], y_tr)

r_val = avaliar_predicao(rf2, X_val[cols], y_val)
r_te  = avaliar_predicao(rf2, X_teste[cols], y_teste)

print(f"validação  F1={r_val['f1']:.4f}  AUC={r_val['roc_auc']:.4f}")
print(f"teste ind. F1={r_te['f1']:.4f}  AUC={r_te['roc_auc']:.4f}  VN={r_te['vn']}")

validação  F1=0.9936  AUC=0.9984
teste ind. F1=0.6659  AUC=0.5867  VN=42


In [67]:
rf3 = RandomForestClassifier(n_estimators=50, class_weight="balanced",
                             random_state=42, n_jobs=1).fit(X_tr[["path_length"]], y_tr)
print("F1 validação com UM atributo:", round(avaliar_predicao(rf3, X_val[["path_length"]], y_val)["f1"], 4))

F1 validação com UM atributo: 0.7491


In [68]:
for atr in ["url_length", "shannon_entropy_hostname", "count_digits"]:
    rf = RandomForestClassifier(n_estimators=50, class_weight="balanced",
                                random_state=42, n_jobs=1).fit(X_tr[[atr]], y_tr)
    f1 = avaliar_predicao(rf, X_val[[atr]], y_val)["f1"]
    print(f"{atr:26s} F1 validação = {f1:.4f}")

url_length                 F1 validação = 0.6379
shannon_entropy_hostname   F1 validação = 0.6313
count_digits               F1 validação = 0.6238


In [69]:
import pandas as pd

df = pd.read_csv("../dados/brutos/dataset_B_05_2020.csv")
print(df.shape)
print(df.columns.tolist()[:10])
df.head(3).T.head(10)

(11430, 89)
['url', 'length_url', 'length_hostname', 'ip', 'nb_dots', 'nb_hyphens', 'nb_at', 'nb_qm', 'nb_and', 'nb_or']


,0,1,2
url,http://www.crestonwood.com/router.php,http://shadetreetechnology.com/V4/validation/a...,https://support-appleld.com.secureupdate.duila...
length_url,37,77,126
length_hostname,19,23,50
ip,0,1,1
nb_dots,3,1,4
nb_hyphens,0,0,1
nb_at,0,0,0
nb_qm,0,0,1
nb_and,0,0,2
nb_or,0,0,0


In [70]:
from pathlib import Path
RAIZ = Path.cwd().parent
BRUTOS = RAIZ / "dados" / "brutos"

df = pd.read_csv(BRUTOS / "dataset_B_05_2020.csv")

In [71]:
print(df.columns.tolist()[-3:])
print(df.iloc[:, -1].value_counts())

['google_index', 'page_rank', 'status']
status
legitimate    5715
phishing      5715
Name: count, dtype: int64


In [72]:
import sys
sys.path.append(str(RAIZ / "src"))

import extracao_lexica
print([n for n in dir(extracao_lexica) if not n.startswith("_")])

['COLUNAS', 'Counter', 'ENCURTADORES', 'TERMOS_SUSPEITOS', 'annotations', 'entropia_shannon', 'extrair_atributos', 'extrair_dataframe', 'ipaddress', 'math', 'pd', 're', 'tldextract', 'urlsplit']


In [73]:
from extracao_lexica import extrair_atributos   # substitua pelo nome real

In [74]:
help(extracao_lexica.extrair_dataframe)

Help on function extrair_dataframe in module extracao_lexica:

extrair_dataframe(urls: 'pd.Series') -> 'pd.DataFrame'
    Aplica a extração a uma série de URLs.

    Retorna um DataFrame com as 37 colunas em ordem canônica, preservando o
    índice da série de entrada para permitir alinhamento com os rótulos.



In [75]:
X_hannousse = extrair_dataframe(df["url"])
y_hannousse = (df["status"] == "phishing").astype(int)

print(X_hannousse.shape)
X_hannousse.head(3).T

(11430, 37)


,0,1,2
url_length,37.000000,77.000000,126.000000
hostname_length,19.000000,23.000000,50.000000
path_length,11.000000,47.000000,20.000000
query_length,0.000000,0.000000,47.000000
longest_token_length,11.000000,32.000000,17.000000
count_dots,3.000000,1.000000,4.000000
count_hyphens,0.000000,0.000000,1.000000
count_underscore,0.000000,0.000000,2.000000
count_slash,3.000000,5.000000,5.000000
count_question,0.000000,0.000000,1.000000


In [76]:
c_h = list(X_hannousse.columns)
c_t = list(X_tr.columns)

print("dim:", len(c_h), len(c_t))
print("só no hannousse:", set(c_h) - set(c_t))
print("só no X_tr:", set(c_t) - set(c_h))
print("mesmo conjunto, ordem diferente:", set(c_h) == set(c_t))

dim: 37 37
só no hannousse: set()
só no X_tr: set()
mesmo conjunto, ordem diferente: True


In [77]:
# como os modelos foram treinados?
print(rf.n_features_in_)

1


In [78]:
import joblib
print(list((RAIZ / "resultados").glob("*")))

[PosixPath('/home/machd-duda14/tcc-phishing/resultados/figuras'), PosixPath('/home/machd-duda14/tcc-phishing/resultados/tabelas')]


In [79]:
for nome in ["lr", "dt", "rf", "xgb"]:   # ajuste para os nomes que você usou
    obj = globals().get(nome)
    print(nome, getattr(obj, "n_features_in_", "ausente"))

lr ausente
dt ausente
rf 1
xgb ausente


In [80]:
MODELOS = RAIZ / "resultados" / "modelos"
MODELOS.mkdir(exist_ok=True)

joblib.dump(modelo, MODELOS / "rf.joblib")

['/home/machd-duda14/tcc-phishing/resultados/modelos/rf.joblib']

In [81]:
from sklearn.base import BaseEstimator

for nome, obj in list(globals().items()):
    if isinstance(obj, BaseEstimator) and not nome.startswith("_"):
        print(nome, type(obj).__name__, getattr(obj, "n_features_in_", "não ajustado"))

modelo XGBClassifier 37
rf2 RandomForestClassifier 32
rf3 RandomForestClassifier 1
rf RandomForestClassifier 1


In [82]:
for nome in ["modelo", "rf2", "rf3", "rf"]:
    obj = globals().get(nome)
    print(nome, getattr(obj, "random_state", "sem atributo"))

modelo 42
rf2 42
rf3 42
rf 42


In [83]:
(MODELOS / "rf.joblib").rename(MODELOS / "xgb.joblib")

PosixPath('/home/machd-duda14/tcc-phishing/resultados/modelos/xgb.joblib')